In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Connect to your database
db_path = r"C:\Users\tania\OneDrive\Documents\2026\2026 Tech Projects\sql-analytics-portfolio\food_economy_analysis\FoodPrices.db"
conn = sqlite3.connect(db_path)

print("✅ Connected to FoodPrices.db successfully!")

✅ Connected to FoodPrices.db successfully!


In [2]:
# Load PPI data only (the comparable metric across countries)
df = pd.read_sql_query("""
    SELECT Area, Item, Element,
           Y2015, Y2016, Y2017, Y2018, Y2019,
           Y2020, Y2021, Y2022, Y2023
    FROM food_prices
    WHERE Element = 'Producer Price Index (2014-2016 = 100)'
    AND Area != ''
    AND Item != ''
""", conn)

print(f"✅ Loaded {len(df):,} rows")
print(f"✅ Columns: {list(df.columns)}")
df.head()

✅ Loaded 13,268 rows
✅ Columns: ['Area', 'Item', 'Element', 'Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020', 'Y2021', 'Y2022', 'Y2023']


,Area,Item,Element,Y2015,Y2016,Y2017,Y2018,Y2019,Y2020,Y2021,Y2022,Y2023
0,Afghanistan,"Almonds, in shell",Producer Price Index (2014-2016 = 100),101.220000,93.360000,89.570000,85.020000,85.740000,86.560000,86.760000,87.060000,89.590000
1,Afghanistan,"Anise, badian, coriander, cumin, caraway, fenn...",Producer Price Index (2014-2016 = 100),101.220000,93.360000,89.570000,85.020000,79.840000,73.460000,67.330000,61.080000,61.980000
2,Afghanistan,Apples,Producer Price Index (2014-2016 = 100),83.580000,129.370000,147.090000,92.010000,102.710000,108.680000,113.680000,115.650000,116.580000
3,Afghanistan,Apricots,Producer Price Index (2014-2016 = 100),98.620000,98.660000,111.670000,107.300000,109.350000,112.680000,116.370000,120.380000,119.330000
4,Afghanistan,Barley,Producer Price Index (2014-2016 = 100),101.220000,93.360000,89.570000,85.020000,81.260000,78.890000,76.500000,74.380000,75.810000


In [3]:
# Convert year columns from text to numbers
year_cols = ['Y2015','Y2016','Y2017','Y2018','Y2019',
             'Y2020','Y2021','Y2022','Y2023']

df[year_cols] = df[year_cols].replace('', None)
df[year_cols] = df[year_cols].astype(float)

print(f"✅ Data cleaned")
print(f"✅ Missing values per year:")
print(df[year_cols].isnull().sum())

✅ Data cleaned
✅ Missing values per year:
Y2015    42
Y2016     2
Y2017     2
Y2018     6
Y2019    59
Y2020    58
Y2021    58
Y2022    60
Y2023    73
dtype: int64


In [4]:
print(f"Countries: {df['Area'].nunique()}")
print(f"Food items: {df['Item'].nunique()}")
print(f"\nSample countries:")
print(df['Area'].unique()[:20])
print(f"\nSample items:")
print(df['Item'].unique()[:20])

Countries: 162
Food items: 234

Sample countries:
<StringArray>
[                     'Afghanistan',                          'Albania',
                          'Algeria',                           'Angola',
              'Antigua and Barbuda',                        'Argentina',
                          'Armenia',                        'Australia',
                          'Austria',                       'Azerbaijan',
                       'Bangladesh',                         'Barbados',
                          'Belarus',                          'Belgium',
                           'Belize',                            'Benin',
                           'Bhutan', 'Bolivia (Plurinational State of)',
           'Bosnia and Herzegovina',                         'Botswana']
Length: 20, dtype: str

Sample items:
<StringArray>
[                                                        'Almonds, in shell',
 'Anise, badian, coriander, cumin, caraway, fennel and juniper berries, raw'

In [5]:
country_trends = df.groupby('Area')[['Y2019', 'Y2023']].mean().dropna()
country_trends['change'] = country_trends['Y2023'] - country_trends['Y2019']
country_trends = country_trends.sort_values('change', ascending=False).head(20)

fig1 = px.bar(
    country_trends,
    x='change',
    y=country_trends.index,
    orientation='h',
    title='Top 20 Countries: Average PPI Rise (2019 → 2023)',
    labels={'change': 'PPI Change (index points)', 'y': 'Country'},
    color='change',
    color_continuous_scale='Reds'
)
fig1.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700
)
fig1.show()

In [6]:
staples = ['Wheat', 'Maize (corn)', 'Rice', 'Sugar cane', 'Milk, Total']

# Filter for items containing staple names
staples_df = df[df['Item'].isin(staples)].copy()

# Melt to long format
staples_melted = staples_df.melt(
    id_vars=['Area', 'Item'],
    value_vars=year_cols,
    var_name='Year',
    value_name='PPI'
).dropna()

staples_melted['Year'] = staples_melted['Year'].str.replace('Y', '').astype(int)

# Average across all countries per item per year
staples_avg = staples_melted.groupby(['Item', 'Year'])['PPI'].mean().reset_index()

fig3 = px.line(
    staples_avg,
    x='Year',
    y='PPI',
    color='Item',
    title='Global Average PPI for Staple Foods (2015-2023)',
    labels={'PPI': 'Average Producer Price Index (2014-2016=100)'},
    markers=True
)
fig3.add_hline(y=100, line_dash='dash', line_color='grey',
               annotation_text='Baseline (2014-2016=100)')
fig3.show()

In [7]:
item_volatility = df.groupby('Item')[['Y2019', 'Y2023']].mean().dropna()
item_volatility['volatility'] = abs(item_volatility['Y2023'] - item_volatility['Y2019'])
item_volatility = item_volatility.sort_values('volatility', ascending=False).head(20)

fig4 = px.bar(
    item_volatility,
    x='volatility',
    y=item_volatility.index,
    orientation='h',
    title='Top 20 Most Volatile Food Items Globally (2019-2023)',
    labels={'volatility': 'Volatility Score (absolute PPI change)', 'y': 'Food Item'},
    color='volatility',
    color_continuous_scale='Oranges'
)
fig4.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700
)
fig4.show()

In [8]:
import os

# Create output folder if it doesn't exist
os.makedirs('visualisations/charts', exist_ok=True)

# Save all charts as interactive HTML
fig1.write_html('visualisations/charts/01_country_price_rise.html')
fig3.write_html('visualisations/charts/02_global_staple_ppi.html')
fig4.write_html('visualisations/charts/03_volatile_food_items.html')

print("✅ All charts saved to visualisations/charts/")

✅ All charts saved to visualisations/charts/
